In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
from analyzer import Analyzer
analyzer = Analyzer()

Analyzer opened


In [3]:
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
from pypfopt.hierarchical_portfolio import HRPOpt

In [4]:
df_for_date = pd.read_csv('../csv_data/stocks/AAPL.csv')

df_pypfopt = pd.DataFrame(
    {
        'Date':df_for_date['Date']
    }
)

# df = pd.read_csv('../csv_data/stocks/AAPL.csv')

stock_lists = [
            'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'META', 'NVDA', 'NFLX', 'AMD',
            'DIS', 'BABA', 'V', 'JPM', 'JNJ', 'WMT', 'PG', 'MA', 'UNH', 'HD',
            'BAC', 'ADBE', 'CRM', 'PYPL', 'INTC', 'VZ', 'KO', 'PEP', 'T',
            'CSCO', 'XOM', 'ABT', 'TMO', 'ACN', 'TXN', 'QCOM', 'COST', 'SBUX',
            'MDT', 'LOW', 'HON', 'UPS', 'IBM', 'GS', 'CAT', 'MMM', 'SLB', 'SPY',
            'LLY', 'ASML', 'MU', 'CVX', 'NVS', 'PM', 'NVO', 'SHEL',
            'KLAC', 'PFE' ,'SPOT', 'BX', 'HOOD', 'GM', 'SE', 'TLT', 'SMH'        ]

for s in stock_lists:
    df = pd.read_csv('../csv_data/stocks/'+s+'.csv')
    df.rename(columns={'Close':s}, inplace=True)
    df_pypfopt = pd.merge(df_pypfopt, df[['Date', s]], on='Date', how='inner')



In [5]:
df_pypfopt.head()

,Date,AAPL,MSFT,GOOGL,AMZN,TSLA,META,NVDA,NFLX,AMD,...,SHEL,KLAC,PFE,SPOT,BX,HOOD,GM,SE,TLT,SMH
0,2021-08-25 00:00:00-04:00,144.706100,289.847198,140.830215,164.959000,237.066666,365.183105,22.136431,54.757999,108.300003,...,32.993248,31.816765,36.464924,226.899994,100.358078,48.980000,47.735401,320.089996,124.420212,129.433456
1,2021-08-26 00:00:00-04:00,143.906235,287.044861,140.197357,165.800003,233.720001,361.208008,21.991932,55.012001,107.269997,...,32.760551,31.791042,36.365147,224.619995,102.339745,46.860001,46.698097,321.769989,124.764809,129.632370
2,2021-08-27 00:00:00-04:00,144.940170,287.649445,142.738281,167.481506,237.306671,369.386200,22.557974,55.891998,111.400002,...,33.491890,32.665478,35.766476,226.020004,104.304588,46.869999,47.831455,321.029999,125.613647,131.975540
3,2021-08-30 00:00:00-04:00,149.348801,291.363586,143.319641,171.078506,243.636673,377.346313,22.609795,56.618000,111.320000,...,33.192703,32.587360,35.889271,233.119995,104.464104,43.639999,47.226364,331.820007,125.941383,132.392746
4,2021-08-31 00:00:00-04:00,148.090607,289.722443,143.425690,173.539505,245.240005,376.077423,22.311771,56.918999,110.720001,...,33.043121,32.382572,35.359688,234.339996,105.572495,44.320000,47.072685,338.320007,125.084122,131.388535


### 12 - 3 - 3
#### 12 months data train - 3 month data test - difference per test: 3 month

In [6]:
# # delta = 1 tahun (12 bulan), 1 array has difference of 3 months
start_end_dates = [
    ('2021-07-01', '2022-07-01'),
    ('2021-10-01', '2022-10-01'),
    ('2022-01-01', '2023-01-01'),
    ('2022-04-01', '2023-04-01'),
    ('2022-07-01', '2023-07-01'),
    ('2022-10-01', '2023-10-01'),
    ('2023-01-01', '2024-01-01'),
    ('2023-04-01', '2024-04-01'),
    ('2023-07-01', '2024-07-01'),
    ('2023-10-01', '2024-10-01'),
    ('2024-01-01', '2025-01-01'),
    ('2024-04-01', '2025-04-01'),
    ('2024-07-01', '2025-07-01'),
    ('2024-10-01', '2025-10-01'),
    ('2025-01-01', '2026-01-01'),
    ('2025-04-01', '2026-04-01')
]

In [7]:
all_weight_data = {}

for (start_date, end_date) in start_end_dates:
    df_semester =  df_pypfopt[(df_pypfopt['Date'] >= start_date) & (df_pypfopt['Date'] <= end_date)].reset_index(drop=True)
    if 'Date' in df_semester.columns:
        df_semester.set_index('Date', inplace=True)
        
    mu = expected_returns.mean_historical_return(df_semester)
    S = risk_models.sample_cov(df_semester)
    
    # Optimize for maximal Sharpe ratio
    ef = EfficientFrontier(mu, S)
    ef.add_constraint(lambda x : x <= 0.20)
    weights = ef.max_sharpe(risk_free_rate=0.05)
    # print(weights)
    res = ef.portfolio_performance(risk_free_rate=0.05, verbose=True)
    expected_annual_return = str(float(res[0]) * 100)+'%'
    annual_volatility = str(float(res[1]) * 100)+'%'
    sharpe_ratio = float(res[2])

    weights_to_buy = {k:weights[k]*100 for k, v in sorted([(v,weights[v]*100) for k,v in enumerate(weights) if weights[v]>0], key=lambda x: x[1], reverse=True)}
    all_weight_data[start_date+'_'+end_date] = {
        'expected_annual_return' : expected_annual_return,
        'annual_volatility' : annual_volatility,
        'sharpe_ratio' : sharpe_ratio,
        'weights_to_buy': weights_to_buy
    }
    print('======')

Expected annual return: 43.3%
Annual volatility: 18.7%
Sharpe Ratio: 2.05
Expected annual return: 35.6%
Annual volatility: 19.0%
Sharpe Ratio: 1.61
Expected annual return: 48.9%
Annual volatility: 22.7%
Sharpe Ratio: 1.94
Expected annual return: 24.8%
Annual volatility: 17.7%
Sharpe Ratio: 1.12
Expected annual return: 92.6%
Annual volatility: 23.1%
Sharpe Ratio: 3.79
Expected annual return: 104.7%
Annual volatility: 19.3%
Sharpe Ratio: 5.18
Expected annual return: 130.1%
Annual volatility: 20.3%
Sharpe Ratio: 6.17
Expected annual return: 97.9%
Annual volatility: 14.4%
Sharpe Ratio: 6.45
Expected annual return: 78.2%
Annual volatility: 13.0%
Sharpe Ratio: 5.66
Expected annual return: 76.2%
Annual volatility: 11.1%
Sharpe Ratio: 6.39
Expected annual return: 93.3%
Annual volatility: 13.1%
Sharpe Ratio: 6.76
Expected annual return: 81.8%
Annual volatility: 13.4%
Sharpe Ratio: 5.72
Expected annual return: 113.2%
Annual volatility: 20.2%
Sharpe Ratio: 5.35
Expected annual return: 130.4%
Annu

In [8]:
all_weight_data

{'2021-07-01_2022-07-01': {'expected_annual_return': '43.34183483186177%',
  'annual_volatility': '18.72675596389831%',
  'sharpe_ratio': 2.0474360271355954,
  'weights_to_buy': {'KO': 20.0,
   'XOM': 20.0,
   'CVX': 20.0,
   'UNH': 18.77989740485301,
   'LLY': 15.60347041873639,
   'SHEL': 4.5603211925392895,
   'PFE': 1.05631098387137}},
 '2021-10-01_2022-10-01': {'expected_annual_return': '35.624055834836106%',
  'annual_volatility': '18.97030165417425%',
  'sharpe_ratio': 1.6143157021489718,
  'weights_to_buy': {'UNH': 20.0,
   'PEP': 20.0,
   'XOM': 20.0,
   'LLY': 20.0,
   'CVX': 20.0,
   'KO': 2e-14}},
 '2022-01-01_2023-01-01': {'expected_annual_return': '48.911083425627396%',
  'annual_volatility': '22.68784898530783%',
  'sharpe_ratio': 1.935444980000673,
  'weights_to_buy': {'XOM': 20.0,
   'LLY': 20.0,
   'CVX': 20.0,
   'NVO': 19.05823752833344,
   'SLB': 10.91606185426806,
   'KO': 5.91071910952854,
   'PM': 4.1149815078699605}},
 '2022-04-01_2023-04-01': {'expected_annual

## Backtest

In [9]:
from backtest import ManualBacktest
from datetime import datetime

In [10]:
backtest_months = 3

In [11]:
mb = ManualBacktest()

for a in all_weight_data:
    allocation = all_weight_data[a]['weights_to_buy']
    print('allocation : ')
    print(allocation)

    start_date_str = a.split('_')[1]
    start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date = mb.delta_date(start_date, months = backtest_months)
    end_date_str = datetime.strftime(mb.delta_date(start_date, months = backtest_months), '%Y-%m-%d')

    print('start_date_str : ', start_date_str)
    print('end_date_str : ', end_date_str)

    df_port = mb.make_portfolio_rebalance(start_date_str, end_date_str, allocation)

    return_backtest = mb.calculate_return_df(df_port, start_date_str, end_date_str)
    std_backtest = mb.calculate_std_df(df_port, start_date_str, end_date_str)
    sharpe_ratio_backtest = return_backtest / std_backtest

    return_spy = mb.calculate_return_benchmark(start_date_str, end_date_str, 'SPY', 'Date')
    std_spy = mb.calculate_std_benchmark(start_date_str, end_date_str, 'SPY', 'Date')
    sharpe_ratio_spy = return_spy / std_spy
    
    return_1n = mb.calculate_return_1n(allocation, df_port)
    
    print('return_backtest : ', return_backtest)
    print('std_backtest : ', std_backtest)

    all_weight_data[a]['backtest'] = {}

    all_weight_data[a]['backtest']['start_date_backtest'] = start_date_str
    all_weight_data[a]['backtest']['end_date_backtest'] = str(end_date)[0:10]
    
    all_weight_data[a]['backtest']['pct_return_backtest'] = float(return_backtest)
    all_weight_data[a]['backtest']['std_backtest'] = float(std_backtest)
    all_weight_data[a]['backtest']['sharpe_ratio_backtest'] = float(sharpe_ratio_backtest)
    
    all_weight_data[a]['backtest']['pct_return_spy'] = float(return_spy)
    all_weight_data[a]['backtest']['std_spy'] = float(std_spy)
    all_weight_data[a]['backtest']['sharpe_ratio_spy'] = float(sharpe_ratio_spy)
    
    all_weight_data[a]['backtest']['pct_return_1n'] = float(return_1n)

    all_weight_data[a]['backtest']['pct_std_backtest'] = float(std_backtest)
    all_weight_data[a]['backtest']['high_water_mark'] = mb.high_water_mark
    all_weight_data[a]['backtest']['drawdown'] = mb.drawdown
    all_weight_data[a]['backtest']['stock_money_profit'] = mb.calculate_underperformed_stocks(allocation, df_port)

    print('all_weight_data[a] : ')
    print(all_weight_data[a])

    print('======')

allocation : 
{'KO': 20.0, 'XOM': 20.0, 'CVX': 20.0, 'UNH': 18.77989740485301, 'LLY': 15.60347041873639, 'SHEL': 4.5603211925392895, 'PFE': 1.05631098387137}
start_date_str :  2022-07-01
end_date_str :  2022-10-01
Total Money end :  96.74125324113102
Total Money start :  100.00000000000007
Total Money end :  339.9685974121094
Total Money start :  361.38470458984375
return_backtest :  -3.2587467588690493
std_backtest :  9.921307618358394
all_weight_data[a] : 
{'expected_annual_return': '43.34183483186177%', 'annual_volatility': '18.72675596389831%', 'sharpe_ratio': 2.0474360271355954, 'weights_to_buy': {'KO': 20.0, 'XOM': 20.0, 'CVX': 20.0, 'UNH': 18.77989740485301, 'LLY': 15.60347041873639, 'SHEL': 4.5603211925392895, 'PFE': 1.05631098387137}, 'backtest': {'start_date_backtest': '2022-07-01', 'end_date_backtest': '2022-10-01', 'pct_return_backtest': -3.2587467588690493, 'std_backtest': 9.921307618358394, 'sharpe_ratio_backtest': -0.3284594011417469, 'pct_return_spy': -5.926124405857394

In [12]:
mean_backtest = []
sr_backtest = []
mean_spy = []
sr_spy = []

for a in all_weight_data:
    mean_backtest.append(all_weight_data[a]['backtest']['pct_return_backtest'])
    std_backtest.append(all_weight_data[a]['backtest']['std_return_backtest'])
    sr_backtest.append(all_weight_data[a]['backtest']['sharpe_ratio_backtest'])

    mean_spy.append(all_weight_data[a]['backtest']['pct_return_spy'])
    std_spy.append(all_weight_data[a]['backtest']['std_return_spy'])
    sr_spy.append(all_weight_data[a]['backtest']['sharpe_ratio_spy'])

In [20]:
print('backtest MVO: ')
print('mean : ', np.mean(mean_backtest))
print('std : ', np.mean(std_backtest))
print('sr : ', np.mean(sr_backtest))

backtest MVO: 
mean :  10.189975314661226
std :  19.14365800469025
sr :  0.9302128014263933


In [21]:
print('SPY: ')
print('mean : ', np.mean(mean_spy))
print('std : ', np.mean(std_spy))
print('sr : ', np.mean(sr_spy))

SPY: 
mean :  4.674380894762109
std :  5.446583003431097
sr :  0.8574999685224636
